# Part 2 — Configure and host the dashboard

[Part 1](01_bioinformatics.ipynb) placed every input in `dashboard/static/`. Here we
1. write a dashboard configuration for the three toy viruses,
2. check it against the data,
3. run the dashboard locally, and
4. package it with Docker and deploy it on a server.

## 0. Setup

In [1]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path
import pandas as pd

if Path.cwd().name == "tutorial":
    os.chdir("..")
REPO = Path.cwd()
sys.path.insert(0, str(REPO))
STATIC = REPO / "dashboard/static"
assert (STATIC / "data/metadata_sequencing.tsv").exists(), "Run Part 1 first."

## 1. How the configuration works
`dashboard/config.py` holds the defaults (the Swiss ReVSeq setup). Rather than editing it, you write **your own settings file** and point the `REVSEQ_CONFIG` environment variable to it. Any setting you leave out keeps its default; the two `*_ALIASES` dictionaries are added to the defaults.

| Setting | What it controls |
|---|---|
| `SITE_TITLE`, `REGION_NAME` | Page header text |
| `DEFAULT_MODULES` | Modules shown on every virus page: `barplot_sequencing`, `barplot_pcr`, `map`, `pileup`, `tree` |
| `STRAIN_CONFIG` | One entry per virus page (see below) |
| `COLOR_BY_STRAIN` | Bar colours per virus |
| `STRAIN_ALIASES` | Extra spellings of virus names → harmonized name |
| `MAP_CONFIG` | GeoJSON file, region-code property, optional basemap |
| `LOCATION_ALIASES` | Extra spellings of locations → region code |
| `MIXED_PAGE_ENABLED` | Show the co-infection page |

Each `STRAIN_CONFIG` entry is keyed by the page's URL slug (`/strain/<slug>/`) and accepts:

| Key | Meaning |
|---|---|
| `label` | Name in the navigation menu |
| `data_name` | Harmonized virus name in the metadata (the page shows rows with this name) |
| `modules` | Per-page overrides of `DEFAULT_MODULES` |
| `pileup_data_prefix` | Pileup files to load: `<prefix>_avg.json` / `<prefix>_indiv.json`; segmented viruses also pick up subtypes `<prefix>-<subtype>_avg.json`. Without a prefix file, substrain files (e.g. `rsv-a_avg.json`) are combined |
| `pileup_segments`, `pileup_default_segment` | Segments (and their GenBank annotation) for segmented viruses |
| `trees` | Auspice trees to embed: `dataset` is the tree file name without `.json` |

Gene annotations for the pileup plots are read from `dashboard/static/annotations/<slug>.gb` (or the per-segment files). The repository ships annotations for common respiratory viruses; add a GenBank file for a new virus.

## 2. Write the configuration
Edit this cell to change the dashboard, then re-run it and the cells below.

In [2]:
%%writefile tutorial/toy_dashboard_config.py
# Dashboard configuration for the tutorial's toy dataset.
# Activate it with:  export REVSEQ_CONFIG=tutorial/toy_dashboard_config.py
# Any setting left out keeps its default from dashboard/config.py.

SITE_TITLE = "ReVSeq tutorial dashboard"
REGION_NAME = "Magic Land (Switzerland)"

# Modules shown on every virus page unless a page overrides them.
DEFAULT_MODULES = {
    "barplot_pcr": True,
    "barplot_sequencing": True,
    "map": True,
    "pileup": True,
    "tree": True,
}

# One entry per virus page. `data_name` must match the harmonized virus name (see STRAIN_ALIASES).
STRAIN_CONFIG = {
    "RSV": {
        "label": "RSV - A/B",
        "data_name": "RSV - A/B",
        # Substrains (RSV-A, RSV-B) are found from rsv-a_avg.json / rsv-b_avg.json.
        "trees": [
            {"title": "RSV A", "dataset": "RSV-A"},
            {"title": "RSV B", "dataset": "RSV-B"},
        ],
    },
    "Influenza_A": {
        "label": "Influenza A",
        "data_name": "Influenza A",
        "pileup_data_prefix": "flu-a",  # finds flu-a-h3n2_avg.json (subtype H3N2)
        "pileup_segments": [
            {"value": segment, "label": segment, "annotation": f"Influenza_A_{segment}.gb"}
            for segment in ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]
        ],
        "pileup_default_segment": "HA",
        "trees": [{"title": "H3N2 segment 4 (HA)", "dataset": "Influenza-A-H3N2-HA"}],
    },
    "Parainfluenza_3": {
        "label": "Parainfluenza 3",
        "data_name": "Parainfluenza 3",
        "pileup_data_prefix": "hpiv-3",
        "trees": [{"title": "", "dataset": "HPIV-3"}],
    },
}

COLOR_BY_STRAIN = {
    "RSV - A/B": "#7B61C9",
    "Influenza A": "#2A9D8F",
    "Parainfluenza 3": "#E9A23B",
}

# Extra spellings of virus names in your metadata -> harmonized name (added to the defaults).
STRAIN_ALIASES = {}  # e.g. {"hRSV-A": "RSV - A/B"}

# Map: the toy samples are Swiss, so we keep the bundled canton map.
# For another country, point `geojson` to your file and `id_property` to the feature
# property holding the codes used in your metadata `location` column.
MAP_CONFIG = {
    "geojson": "dashboard/static/swiss_cantons.geojson",
    "id_property": "canton",
    "name_property": "NAME",
}

# Extra location spellings -> region code (added to the defaults). Case and accents are ignored.
LOCATION_ALIASES = {"Kanton Bern": "BE"}

Overwriting tutorial/toy_dashboard_config.py


In [3]:
os.environ["REVSEQ_CONFIG"] = "tutorial/toy_dashboard_config.py"

## 3. Check the configuration against the data
Three things commonly go wrong with a new dataset: virus names don't map to a configured page, locations don't map to a region of the map, and we have missing files. This cell checks all three with the dashboard's own harmonization functions.

In [4]:
import importlib
from dashboard import config
importlib.reload(config)  # pick up edits made in step 2

seq = pd.read_csv(STATIC / "data/metadata_sequencing.tsv", sep="\t")
pcr = pd.read_csv(STATIC / "data/metadata_pcr.tsv", sep="\t")
names = pd.concat([seq.virus_identified.rename("name").to_frame().assign(source="sequencing"),
                   pcr.virus_identified_pcr.rename("name").to_frame().assign(source="pcr")]).drop_duplicates()
names["page"] = [config.harmonize_detected_strain(n, s)[0] for n, s in zip(names.name, names.source)]
print("Virus names -> page (None = dropped, add it to STRAIN_ALIASES):")
display(names)

locations = pd.DataFrame({"location": seq.location.unique()})
locations["region"] = locations.location.map(config.harmonize_location)
print("Locations without a map region (add them to LOCATION_ALIASES):", list(locations[locations.region.isna()].location))

missing = []
for slug, page in config.STRAIN_CONFIG.items():
    annotations = [s["annotation"] for s in page.get("pileup_segments", [])] or [page.get("pileup_annotation", f"{slug}.gb")]
    missing += [f"annotations/{a}" for a in annotations if not (STATIC / "annotations" / a).exists()]
    missing += [f"auspice/{t['dataset']}.json" for t in page.get("trees", []) if not (STATIC / "auspice" / f"{t['dataset']}.json").exists()]
print("Missing files:", missing or "none")

Virus names -> page (None = dropped, add it to STRAIN_ALIASES):


,name,source,page
0,rsv a,sequencing,RSV - A/B
7,rsv b,sequencing,RSV - A/B
11,influenza a h3n2,sequencing,Influenza A
16,parainfluenza 3,sequencing,Parainfluenza 3
0,RSV - A/B,pcr,RSV - A/B
11,Influenza A,pcr,Influenza A
16,Parainfluenza 3,pcr,Parainfluenza 3


Locations without a map region (add them to LOCATION_ALIASES): []
Missing files: none


Try it: remove `"Kanton Bern": "BE"` from `LOCATION_ALIASES` in step 2 and re-run — the co-infected sample's location is reported as unmapped and would be left off the map.

## 4. Run the dashboard locally
The dashboard is two web servers:
- **Auspice** (`auspice-revseq`, a Nextstrain viewer) serves the phylogenetic trees on port 4001;
- **Django** serves the pages on port 8000 and embeds the trees from `NEXTSTRAIN_BASE_URL`.

The cell starts both in the background and waits until each one answers; if a server fails, it stops and shows the end of that server's log (`tutorial/workdir/*.log`). In GitHub Codespaces the cell also makes port 4001 public, since the pages embed the trees from it. The first run installs Auspice with `npm ci` (about a minute), at the versions pinned in `package-lock.json`. From a terminal you would run the same two commands, shown in the `README`.

In [5]:
LOGS = REPO / "tutorial/workdir"

def public_url(port):
    # GitHub Codespaces exposes ports on their own domain; elsewhere it is localhost.
    if "CODESPACE_NAME" in os.environ:
        return f"https://{os.environ['CODESPACE_NAME']}-{port}.{os.environ['GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN']}"
    return f"http://127.0.0.1:{port}"

def start(name, command, port, path="/", timeout=300):
    """Start a server in the background and wait until it answers; on failure show the end of its log."""
    log = LOGS / f"{name}.log"
    process = subprocess.Popen(command, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(timeout):
        if process.poll() is not None:  # e.g. "port already in use": stop the old servers or restart the kernel
            raise RuntimeError(f"{name} stopped. End of {log}:\n" + "\n".join(log.read_text().splitlines()[-15:]))
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}{path}", timeout=120)
            return process
        except urllib.error.HTTPError as error:
            process.terminate()
            raise RuntimeError(f"{name} answered HTTP {error.code}; see {log}") from None
        except OSError:
            time.sleep(1)
    process.terminate()
    raise TimeoutError(f"{name} did not answer within {timeout} s; see {log}")

env = {**os.environ,
       "HOST": "0.0.0.0", "PORT": "4001",
       "NEXTSTRAIN_BASE_URL": public_url(4001),
       "DJANGO_ALLOWED_HOSTS": "localhost,127.0.0.1,.app.github.dev"}
if not (REPO / "node_modules/.bin/auspice").exists():
    subprocess.run(["npm", "ci", "--no-audit", "--no-fund", "--loglevel=error"], check=True)
django = start("django", [sys.executable, "manage.py", "runserver", "0.0.0.0:8000"], 8000, "/home/")
auspice = start("auspice", ["npx", "auspice", "view", "--datasetDir", "dashboard/static/auspice"], 4001)
if "CODESPACE_NAME" in os.environ:
    # Pages embed the trees from port 4001, which must be reachable without a GitHub sign-in.
    subprocess.run(["gh", "codespace", "ports", "visibility", "4001:public", "-c", os.environ["CODESPACE_NAME"]])
print("Dashboard:", public_url(8000) + "/home/")


Dashboard: http://127.0.0.1:8000/home/


Open the link printed above. The first visit to a page renders its plots, later visits reuse them until the data or configuration change.

| Home: time series and map | RSV page: substrains, sequencing depth and trees |
|---|---|
| ![home](images/home.jpg) | ![rsv](images/rsv_page.jpg) |

### Finding a coverage dropout
The **Sequencing depth** panel of a virus page shows how well each part of the genome was covered, with the annotated genes below. In the toy data three RSV-A samples lost the same stretch of the G gene. To find them on the RSV page:
1. **All samples**: median depth and interquartile range over every RSV sample. A sharp dip that drops the lower quartile is the first sign of a problem.
2. **Substrains**: the same summary per substrain (pick it in the drop-down). The dip is in RSV-A only, around 4.9–5.3 kb, inside the G gene.
3. **Individuals**: one line per sample. Over the G gene three RSV-A lines fall to the bottom of the plot and lie on top of each other; hover over them to read the sample ID, or drag a box to zoom in.

Because the gap sits at the same place in several samples, it points to the assay (for example a primer that no longer matches the circulating strain) rather than to individual poor samples. Positions below the minimum depth are masked as `N` in the consensus (Part 1, step 2), so these genomes are also incomplete in the tree.

![Substrains view, RSV-A: the median and interquartile range drop to zero inside the G gene](images/dropout_substrain.jpg)

![Individuals view: hovering over the gap shows the sample ID](images/dropout_individual.jpg)

The same check in code, on the file the panel reads: list each sample's stretches of at least 100 bp without any reads, and the gene they fall in.

In [6]:
from Bio import SeqIO

indiv = json.loads((STATIC / "data/pileup/rsv-a_indiv.json").read_text())
genes = [(f.qualifiers["gene"][0], int(f.location.start), int(f.location.end))
         for f in SeqIO.read(STATIC / "annotations/RSV.gb", "genbank").features if f.type == "gene"]

gaps = []
for sample, depth in indiv["samples"].items():
    zero = pd.Series([d == 0 for d in depth], index=indiv["positions"])
    runs = (zero != zero.shift()).cumsum()[zero]  # label consecutive zero-depth positions
    for _, run in runs.groupby(runs):
        start, end = run.index[0], run.index[-1]
        if end - start >= 100:
            gene = ", ".join(g for g, s, e in genes if s < end and e > start)
            gaps.append({"sample": sample, "start": start, "end": end, "gene": gene})
pd.DataFrame(gaps)

,sample,start,end,gene
0,TOY004,4901,5250,G
1,TOY007,4900,5250,G
2,TOY015,4084,4316,"M, SH"
3,TOY015,4901,5250,G
4,TOY019,1776,1914,N


The shared G-gene gap stands out. The shorter gaps in single samples are ordinary dropouts of one sample's library; they are also visible in the **Individuals** view, but only on one line each.

### How fast is it?
Pages are rendered on the first request and cached as HTML until an input changes. This cell measures both cases for every page; it also gives you a first benchmark for your own dataset.

In [7]:
pages = ["/home/", *[f"/strain/{slug}/" for slug in config.STRAIN_CONFIG], "/mixed/"]

def load_seconds(page):
    start = time.perf_counter()
    urllib.request.urlopen("http://127.0.0.1:8000" + page).read()
    return round(time.perf_counter() - start, 2)

for html in [*STATIC.glob("*.html"), *STATIC.glob("barplots/*.html"), *STATIC.glob("pileup_html/*.html")]:
    html.unlink()  # drop rendered plots to force a cold start
pd.DataFrame({"page": pages,
              "first load (s)": [load_seconds(p) for p in pages],
              "cached (s)": [load_seconds(p) for p in pages]})

,page,first load (s),cached (s)
0,/home/,0.20,0.00
1,/strain/RSV/,0.34,0.05
2,/strain/Influenza_A/,0.60,0.16
3,/strain/Parainfluenza_3/,0.17,0.01
4,/mixed/,0.15,0.11


## 5. Using your own map
The map works with any GeoJSON whose features carry a code that also appears in your metadata `location` column. Inspect a feature to find which property holds that code:

In [8]:
geojson = json.loads((REPO / config.MAP_CONFIG["geojson"]).read_text())
pd.DataFrame([f["properties"] for f in geojson["features"]]).head(3)

,canton,NAME
0,GE,Genève
1,TG,Thurgau
2,VS,Valais


Here `canton` holds the code (`id_property`) and `NAME` the label shown on hover (`name_property`). For another country, download its regions as GeoJSON (for example from [geoBoundaries](https://www.geoboundaries.org)), save it under `dashboard/static/`, and set:

```python
MAP_CONFIG = {
    "geojson": "dashboard/static/my_regions.geojson",
    "id_property": "shapeISO",       # e.g. "FR-IDF"
    "name_property": "shapeName",
}
LOCATION_ALIASES = {"Paris": "FR-IDF"}  # spellings in your metadata -> codes in the GeoJSON
```
Locations that cannot be mapped are still counted in the time series; they are only left off the map.

By default the map draws only your regions, so it needs no outside service. To show a street map behind them, add `"style": "open-street-map"` with a `"center"` and `"zoom"`. Those tiles come from OpenStreetMap's volunteer servers, whose [usage policy](https://operations.osmfoundation.org/policies/tiles/) may block busy or unrecognised sites, so use a commercial tile provider for a public dashboard.

Stop the local servers before moving on:

In [9]:
for process in (django, auspice):
    process.terminate()

## 6. Package with Docker
The repository has two images: the dashboard (`Dockerfile`) and the tree viewer (`Dockerfile_nextstrain`). Both copy `dashboard/static/`, so **the data are baked into the images** — rebuild after updating the data. From a terminal at the repository root:

```bash
export REVSEQ_CONFIG=tutorial/toy_dashboard_config.py
docker compose up --build -d
```

Open <http://localhost:8000/home/> (trees on <http://localhost:4000>). Stop with `docker compose down`.

## 7. Deploy on a server
1. **Build and push versioned images** from your machine:
   ```bash
   export REVSEQ_TAG=2026.10.28-1 DOCKERHUB_USER=<your-user>
   docker build -f Dockerfile -t $DOCKERHUB_USER/revseq-dashboard:$REVSEQ_TAG .
   docker build -f Dockerfile_nextstrain -t $DOCKERHUB_USER/revseq-nextstrain:$REVSEQ_TAG .
   docker push $DOCKERHUB_USER/revseq-dashboard:$REVSEQ_TAG
   docker push $DOCKERHUB_USER/revseq-nextstrain:$REVSEQ_TAG
   ```
2. **On the server**, copy `docker-compose.deploy.yml`, replace `<username>` and your domain in it, and create a `.env` next to it:
   ```text
   REVSEQ_TAG=2026.10.28-1
   DJANGO_SECRET_KEY=<long random string>
   REVSEQ_CONFIG=tutorial/toy_dashboard_config.py
   ```
   Then `docker compose -f docker-compose.deploy.yml pull && docker compose -f docker-compose.deploy.yml up -d`.
3. **Put a reverse proxy in front** (HTTPS, one domain). The dashboard expects Auspice under `/nextstrain/` (`NEXTSTRAIN_BASE_URL=/nextstrain`):

   | Path | Route to |
   |---|---|
   | `/nextstrain/` | Auspice, port 4000 (prefix removed) |
   | `/dist/`, `/charon/` | Auspice, port 4000 |
   | everything else | Django, port 8000 |

To update the data later: re-run Part 1 with the new BAMs, rebuild the images with a new tag, and pull them on the server.

## Summary
| Step | Command / file |
|---|---|
| Dashboard settings | your `REVSEQ_CONFIG` file (here `tutorial/toy_dashboard_config.py`) |
| Check settings vs data | step 3 of this notebook |
| Run locally | `npm ci` + `npx auspice view --datasetDir dashboard/static/auspice` + `python manage.py runserver` |
| Run with Docker | `docker compose up --build` |
| Deploy | versioned images + `docker-compose.deploy.yml` + reverse proxy |